# MLflow experiment Tracking

In [15]:
import pandas as pd  
import mlflow
import os
import joblib
import json
from sklearn.metrics import (accuracy_score, f1_score, recall_score)
from sklearn.preprocessing import OneHotEncoder, LabelEncoder

In [2]:
print("Tracking URI:", mlflow.get_tracking_uri())

Tracking URI: file:///c:/Users/ayata/OneDrive/Healthcare%20ml%20project/notebooks/mlruns


In [3]:
mlflow.set_tracking_uri("sqlite:///../mlflow.db")
print("Tracking URI:", mlflow.get_tracking_uri())

Tracking URI: sqlite:///../mlflow.db


In [4]:
# set the experiment
mlflow.set_experiment("healthcare-risk-classification")
print("experiment created")

experiment created


In [5]:
risk_xgb_model = joblib.load("../models/risk_model.joblib")
claim_xgb_model = joblib.load("../models/claim_model.joblib")
print("models loaded")
print("risk model :",type(risk_xgb_model))
print("claim model :",type(claim_xgb_model))

models loaded
risk model : <class 'sklearn.pipeline.Pipeline'>
claim model : <class 'sklearn.pipeline.Pipeline'>


In [6]:
# loading the dataset
df = pd.read_csv("../outputs/model_table.csv",parse_dates=["registration_date","visit_date","billing_date"])
print("data loaded")
print("Shape: ",df.shape)
df.head()

data loaded
Shape:  (25000, 30)


,patient_id,age,gender,city,insurance_provider,chronic_flag,registration_date,visit_id,visit_date,department,...,risk_numeric,claim_numeric,is_rejected,days_since_registration,visit_frequency,avg_los_per_patient,provider_rejection_rate,visit_month,visit_dayofweek,high_cost_visit_flag
0,2,15,F,Mumbai,CareOne,0,2025-12-27,8,2026-01-01,General,...,NaN,2,1,5,4,21.120000,0.256876,1,3,False
1,12,3,M,Bangalore,CareOne,0,2025-08-13,65,2026-01-01,ICU,...,2.0,2,1,141,8,23.750000,0.256876,1,3,True
2,129,44,M,Pune,MediCareX,1,2025-07-20,651,2026-01-01,ICU,...,2.0,1,0,165,3,32.460000,0.242556,1,3,True
3,133,47,F,Delhi,CareOne,1,2025-11-02,670,2026-01-01,General,...,NaN,0,0,60,3,30.056667,0.256876,1,3,False
4,139,14,F,Chennai,SecureLife,1,2025-02-05,706,2026-01-01,Cardiology,...,NaN,0,0,330,9,29.030000,0.157496,1,3,True


In [8]:
# loading feature schema
with open("../outputs/feature_schema.json","r") as f:
    schema = json.load(f)

risk_features = schema["risk_model_features"]
claim_features = schema["claim_model_features"]
risk_target = schema["risk_target"]
claim_target = schema["claim_target"]

print("schema loaded")
print(f"risk features : {len(risk_features)}")
print(f"claim features : {len(claim_features)}")

schema loaded
risk features : 14
claim features : 18


In [9]:
# creating seperate datasets
risk_df = df.copy()
claim_df = df.copy()

In [16]:
le = LabelEncoder()
risk_df["risk_score_encoded"] = le.fit_transform(risk_df["risk_score"])
risk_target = "risk_score_encoded"

In [17]:
le = LabelEncoder()
claim_df["claim_status_encoded"] = le.fit_transform(claim_df["claim_status"])
claim_target = "claim_status_encoded"

In [18]:
# time based split
risk_df = risk_df.sort_values("visit_date").reset_index(drop=True)

split_idx = int(len(risk_df) * 0.8)

risk_train = risk_df.iloc[:split_idx].copy()
risk_test = risk_df.iloc[split_idx:].copy()

X_train_risk = risk_train[risk_features]
X_test_risk = risk_test[risk_features]

y_train_risk = risk_train[risk_target]
y_test_risk = risk_test[risk_target]

print("Risk Train shape: ",X_train_risk.shape)
print("Risk Test shape: ",X_test_risk.shape)
print("Risk Train period: ",risk_train["visit_date"].min().date(), "=>",risk_train["visit_date"].max().date())
print("Risk Test period: ",risk_test["visit_date"].min().date(), "=>",risk_test["visit_date"].max().date())

Risk Train shape:  (20000, 14)
Risk Test shape:  (5000, 14)
Risk Train period:  2025-01-21 => 2026-01-02
Risk Test period:  2026-01-02 => 2026-01-20


In [19]:
# Time based split - for claim
claim_df = claim_df.sort_values("billing_date").reset_index(drop=True)

split_idx = int(len(claim_df)*0.8)

claim_train = claim_df.iloc[:split_idx].copy()
claim_test = claim_df.iloc[split_idx:].copy()

X_train_claim = claim_train[claim_features]
X_test_claim = claim_test[claim_features]

y_train_claim = claim_train[claim_target]
y_test_claim = claim_test[claim_target]

print("Claim train shape: ",X_train_claim.shape)
print("Claim test shape: ",X_test_claim.shape)
print("Claim train period: ",claim_train['billing_date'].min().date(),"=>",claim_train['billing_date'].max().date())
print("Claim test period: ",claim_test['billing_date'].min().date(),"=>",claim_test['billing_date'].max().date())

Claim train shape:  (20000, 18)
Claim test shape:  (5000, 18)
Claim train period:  2025-01-28 => 2026-01-17
Claim test period:  2026-01-17 => 2026-01-20


In [23]:
# Log risk model
with mlflow.start_run(run_name="XGBoost-Risk") as run:
    mlflow.log_params({
        "model":"XGBClassifier",
        "objective":"multi:softmax",
        "num_class":3,
        "n_estimators":500,
        "max_depth":6,
        "learning_rate":0.05,
        "subsample":0.8,
        "colsample_bytree":0.8,
        "random_state":42,
        "verbosity":0,
        "evaluation_data":"risk_test_only",
        "split_strategy":"time_based_split",
        "split_column":"visit_date"
    })
    pred_risk = risk_xgb_model.predict(X_test_risk)

    acc_risk = accuracy_score(y_test_risk,pred_risk)
    f1_risk = f1_score(y_test_risk,pred_risk,average="weighted")
    high_recall = recall_score(
    y_test_risk,
    pred_risk,
    labels=[0],
    average=None,
    zero_division=0
    )[0]

    #logging metrics
    mlflow.log_metric("accuracy",acc_risk)
    mlflow.log_metric("weighted_f1",f1_risk)
    mlflow.log_metric("high_risk_recall",high_recall)

    # logging model
    mlflow.sklearn.log_model(
        sk_model = risk_xgb_model,
        artifact_path="model"
    )

    # getting the run id
    risk_run_id = run.info.run_id

    print("XGBoost Risk Model logged")
    print(f"Accuracy            :{acc_risk:.4f}")
    print(f"Weighted f1         :{f1_risk:.4f}")
    print(f"High Risk Recall    :{high_recall:.4f}")
    print(f"Run ID              :{risk_run_id}")


c:\Users\ayata\anaconda33\envs\final_project\Lib\site-packages\_distutils_hack\__init__.py:17: UserWarning: Distutils was imported before Setuptools, but importing Setuptools also replaces the `distutils` module in `sys.modules`. This may lead to undesirable behaviors or errors. To avoid these issues, avoid using distutils directly, ensure that setuptools is installed in the traditional way (e.g. not an editable install), and/or make sure that setuptools is always imported before distutils.
  warnings.warn(
c:\Users\ayata\anaconda33\envs\final_project\Lib\site-packages\_distutils_hack\__init__.py:32: UserWarning: Setuptools is replacing distutils. Support for replacing an already imported distutils is deprecated. In the future, this condition will fail. Register concerns at https://github.com/pypa/setuptools/issues/new?template=distutils-deprecation.yml
  warnings.warn(
2026/10/20 23:29:09 WARNING mlflow.models.model: Input example should be provided to infer model signature if the mod

XGBoost Risk Model logged
Accuracy            :0.9568
Weighted f1         :0.9569
High Risk Recall    :0.9392
Run ID              :8363ad55e3044b3dbc7770d4135d1218


In [24]:
print(mlflow.get_tracking_uri())

sqlite:///../mlflow.db
